In [2]:
"""
NAP — Simulation Study II  (nonlinear regression)
==================================================
Data-generating model:
    y = 7*x2/(1+x1^2) + sin(x3*x4) + 2*x5 + ε,   ε ~ N(0,1)
    x1,...,x5 i.i.d. N(0,1),  p=5 active inputs

Architecture : FNN  5 → 20 → 20 → 1   (2 hidden layers, 20 neurons each)
Activation   : Swish  f(x) = x·σ(x)
Initialisation:
    μ    ~ U(-0.6, 0.6)          (thesis spec for simulation)
    ρ    = -6  → σ = softplus(-6) ≈ 0.0025
    γ_lj = logit(0.999) ≈ 6.9   (close to 1, fully connected at epoch 0)
Training     : Adam, lr=5e-3, FULL BATCH, 10 000 epochs, S=1 MC sample
Reporting    : mean ± sd of RMSE and median sparsity from last 1000 epochs
               sampled every 10 epochs  → 100 checkpoints

Full NAP pipeline:
    Phase 1 — Bayesian Lottery Ticket (IMP)
              Structured node-level IMP on full dense network.
              Each round: full-batch train 2000 epochs, prune weakest
              nodes by L1 row-norm of μ, rewind to epoch-0.
    Phase 2 — SS-IG with soft Gumbel-Softmax (10 000 epochs)
              Run on the winning-ticket topology, prior recalibrated
              to ticket dimensions. S=1 MC sample, full-batch.
    Phase 3 — Poisson-Binomial optimal layer size
    Phase 4 — Neural atom selection
    Phase 5 — Fine-tune sparse BNN (10 000 epochs, epoch-0 rewind)

Seeds  : 42 everywhere
Device : CUDA if available, else CPU
"""

import math, random, os
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {DEVICE}")

# ── plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi"       : 150,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "font.size"        : 11,
})
_C = {
    "ssig"   : "#4C72B0",
    "ft"     : "#27AE60",
    "l1"     : "#8E44AD",
    "l2"     : "#16A085",
    "rmse"   : "#E07B39",
    "bg"     : "#F8F9FA",
    "grid"   : "#DEE2E6",
}


# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA GENERATION
# ─────────────────────────────────────────────────────────────────────────────

def make_dataset(n: int, seed: int = SEED):
    """
    Generate n samples from:
        y = 7*x2/(1+x1^2) + sin(x3*x4) + 2*x5 + ε
    Returns X (n,5), y (n,1) as float32 tensors.
    """
    rng = np.random.default_rng(seed)
    X   = rng.standard_normal((n, 5)).astype(np.float32)
    eps = rng.standard_normal(n).astype(np.float32)
    x1, x2, x3, x4, x5 = X[:, 0], X[:, 1], X[:, 2], X[:, 3], X[:, 4]
    y   = (7 * x2 / (1 + x1**2) + np.sin(x3 * x4) + 2 * x5 + eps).astype(np.float32)
    return torch.from_numpy(X), torch.from_numpy(y).unsqueeze(1)


def get_tensors():
    X_train, y_train = make_dataset(3000, seed=SEED)
    X_test,  y_test  = make_dataset(1000, seed=SEED + 1)
    print(f"[data] train={X_train.shape}  test={X_test.shape}")
    return (X_train.to(DEVICE), y_train.to(DEVICE),
            X_test.to(DEVICE),  y_test.to(DEVICE))


# ─────────────────────────────────────────────────────────────────────────────
# 2.  NETWORK ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────────

class SwishActivation(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)


class BayesianLinear(nn.Module):
    """
    Variational Bayesian linear layer (regression variant).

    Weight prior  : W_j | v_j ~ (1-v_j)*δ_0 + v_j * N(0, σ0² I)
    Variational q : W_j | v_j ~ N(μ_j, diag(σ_j²)),  σ_j = softplus(ρ_j)
    Inclusion     : v_j ~ Bernoulli(π̃_j),  π̃_j = sigmoid(logit_π_j)

    Regression init (thesis spec):
        μ    ~ U(-0.6, 0.6)
        ρ    = -6  (constant)
        γ_lj = logit(0.999)  → π̃ ≈ 0.999 at epoch 0
    """
    def __init__(self, in_features: int, out_features: int,
                 sigma0: float = 1.0):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.sigma0       = sigma0

        # μ and ρ stored as (out, in+1): weights + bias concatenated
        self.mu       = nn.Parameter(torch.empty(out_features, in_features + 1))
        self.rho      = nn.Parameter(torch.empty(out_features, in_features + 1))
        self.logit_pi = nn.Parameter(torch.empty(out_features))
        self._init_params()

    def _init_params(self):
        # μ ~ U(-0.6, 0.6)   
        nn.init.uniform_(self.mu, -0.6, 0.6)
        # ρ = -6 → σ = softplus(-6) ≈ 0.0025
        nn.init.constant_(self.rho, -6.0)
        # γ close to 1: logit(0.999)
        nn.init.constant_(self.logit_pi, math.log(0.999 / 0.001))

    def snapshot_epoch0(self):
        self._mu0       = self.mu.data.clone()
        self._rho0      = self.rho.data.clone()
        self._logit_pi0 = self.logit_pi.data.clone()

    def rewind_to_epoch0(self):
        self.mu.data.copy_(self._mu0)
        self.rho.data.copy_(self._rho0)
        self.logit_pi.data.copy_(self._logit_pi0)

    @property
    def pi_tilde(self) -> torch.Tensor:
        return torch.sigmoid(self.logit_pi)

    def gumbel_softmax_gate(self, tau: float = 0.5) -> torch.Tensor:
        alpha  = torch.zeros_like(self.logit_pi).uniform_().clamp(1e-6, 1 - 1e-6)
        gumbel = torch.log(alpha) - torch.log(1 - alpha)
        return torch.sigmoid((self.logit_pi + gumbel) / tau)

    def forward(self, x: torch.Tensor, tau: float = 0.5,
                n_samples: int = 1) -> torch.Tensor:
        """
        Reparameterised forward pass.
        n_samples=1 during training (thesis: S=1).
        Returns mean over samples: shape (batch, out_features).
        """
        sigma = F.softplus(self.rho)                  # (out, in+1)
        v     = self.gumbel_softmax_gate(tau)         # (out,)

        outs = []
        for _ in range(n_samples):
            eps      = torch.randn_like(self.mu)
            w_sample = self.mu + sigma * eps           # (out, in+1)
            w        = w_sample[:, :-1]               # (out, in)
            b        = w_sample[:, -1]                # (out,)
            out      = F.linear(x, w, b) * v.unsqueeze(0)
            outs.append(out)

        return torch.stack(outs, 0).mean(0)           # (batch, out)

    # ── KL terms ─────────────────────────────────────────────────────────────
    def kl_v(self, prior_pi: float) -> torch.Tensor:
        pi = self.pi_tilde.clamp(1e-6, 1 - 1e-6)
        p  = torch.tensor(prior_pi, device=pi.device).clamp(1e-6, 1 - 1e-6)
        return (pi * (torch.log(pi) - torch.log(p)) +
                (1 - pi) * (torch.log(1 - pi) - torch.log(1 - p))).sum()

    def kl_weights(self) -> torch.Tensor:
        sigma   = F.softplus(self.rho)
        sigma0  = torch.tensor(self.sigma0, device=self.mu.device)
        kl_each = 0.5 * (
            (sigma / sigma0).pow(2) + (self.mu / sigma0).pow(2)
            - 1 - 2 * torch.log(sigma / sigma0)
        ).sum(dim=1)
        return (self.pi_tilde * kl_each).sum()

    def effective_sparsity(self, dense_size: int) -> float:
        """Fraction of ORIGINAL dense neurons that are inactive (π̃_j ≤ 0.5).
        Always uses `dense_size` (the original hidden width) as the denominator,
        regardless of how many neurons this layer currently has.
        """
        n_inactive = (self.pi_tilde.detach() <= 0.5).sum().item()
        return n_inactive / dense_size

    @property
    def mean_pi(self) -> float:
        return self.pi_tilde.detach().mean().item()


class RegressionNAP(nn.Module):
    """
    FNN  p → 20 → 20 → 1  for regression.
    Hidden layers: BayesianLinear + Swish.
    Output layer : plain deterministic linear (scalar output).
    σ_ε²  is a learned scalar (log-parameterised for positivity).
    """
    def __init__(self, p_in: int = 5, hidden: int = 20,
                 sigma0: float = 1.0):
        super().__init__()
        self.layer1  = BayesianLinear(p_in,  hidden, sigma0)
        self.layer2  = BayesianLinear(hidden, hidden, sigma0)
        self.out     = nn.Linear(hidden, 1)
        self.act     = SwishActivation()
        # log σ_ε — learned noise variance (initialised at log(1) = 0)
        self.log_sigma_eps = nn.Parameter(torch.zeros(1))

    def forward(self, x: torch.Tensor, tau: float = 0.5,
                n_samples: int = 1) -> torch.Tensor:
        x = self.act(self.layer1(x, tau=tau, n_samples=n_samples))
        x = self.act(self.layer2(x, tau=tau, n_samples=n_samples))
        return self.out(x)                            # (batch, 1)

    @property
    def sigma_eps(self) -> torch.Tensor:
        return torch.exp(self.log_sigma_eps).clamp(1e-3, 10.0)

    def snapshot_epoch0(self):
        self.layer1.snapshot_epoch0()
        self.layer2.snapshot_epoch0()

    def rewind_to_epoch0(self):
        self.layer1.rewind_to_epoch0()
        self.layer2.rewind_to_epoch0()
        nn.init.uniform_(self.out.weight, -0.6, 0.6)
        nn.init.zeros_(self.out.bias)
        nn.init.zeros_(self.log_sigma_eps)

    def elbo(self, x: torch.Tensor, y: torch.Tensor,
             prior_pi1: float, prior_pi2: float,
             n_data: int, kl_weight: float = 1.0,
             tau: float = 0.5,
             n_mc: int = 1) -> torch.Tensor:
        """
        Gaussian log-likelihood ELBO for regression.

        log P(y|x,θ) = -n/2 log(2π σ_ε²) - 1/(2σ_ε²) Σ(y - ŷ)²
        """
        y_hat    = self.forward(x, tau=tau, n_samples=n_mc)  # (n, 1)
        sigma2   = self.sigma_eps ** 2
        n        = x.size(0)

        log_like = (
            - 0.5 * n * torch.log(2 * torch.tensor(math.pi) * sigma2)
            - 0.5 / sigma2 * ((y - y_hat) ** 2).sum()
        )

        kl  = self.layer1.kl_v(prior_pi1) + self.layer1.kl_weights()
        kl += self.layer2.kl_v(prior_pi2) + self.layer2.kl_weights()

        # scale likelihood to full dataset size (full-batch: n == n_data here)
        return (n_data / n) * log_like - kl_weight * kl

    @torch.no_grad()
    def predict_mc(self, x: torch.Tensor,
                   n_samples: int = 50) -> torch.Tensor:
        """
        Returns stacked MC predictions. Shape: (n_samples, batch, 1).
        """
        self.eval()
        return torch.stack(
            [self.forward(x, tau=0.5, n_samples=1) for _ in range(n_samples)],
            dim=0
        )

    @torch.no_grad()
    def rmse(self, x: torch.Tensor, y: torch.Tensor,
             n_samples: int = 50) -> float:
        preds = self.predict_mc(x, n_samples).mean(0)    # (n, 1)
        return float(((preds - y) ** 2).mean().sqrt().item())


# ─────────────────────────────────────────────────────────────────────────────
# 3.  PRIOR CALIBRATION  (Corollary 4.5, Jantre et al. 2022)
# ─────────────────────────────────────────────────────────────────────────────

def compute_prior_pi(k_in: int, k_out: int, k_next: int,
                     n: int, L: int = 2,
                     B_l: float | None = None) -> float:
    if B_l is None:
        B_l = float(k_out + 1)

    def U_m(k):
        return (L + 1)**2 * (math.log(n) + math.log(L + 1)
                             + math.log(k + 1) + math.log(k + 1))

    U_sum  = sum(U_m(k) for k in [k_in, k_out])
    V_l    = (B_l**2 / (k_out + 1) + math.log(max(B_l, 1e-8))
              + L + math.log(k_out + 1) + math.log(n)
              + math.log(U_sum + 1e-8))
    log_pi = -(math.log(k_next) + 1.0 * (k_out + 1) * V_l)
    return float(np.clip(math.exp(max(log_pi, -20.0)), 1e-6, 1 - 1e-6))


# ─────────────────────────────────────────────────────────────────────────────
# 4.  SPARSITY SNAPSHOT
# ─────────────────────────────────────────────────────────────────────────────

def snapshot_sparsity(model: RegressionNAP, dense_size: int) -> dict:
    """Compute per-layer sparsity/activity always relative to `dense_size`
    (the original hidden-layer width of the dense architecture).
    This makes all snapshots comparable across IMP, SS-IG, and fine-tune
    phases, regardless of how many neurons the current model has."""
    snap = {}
    for lname in ["layer1", "layer2"]:
        pi        = getattr(model, lname).pi_tilde.detach().cpu().numpy()
        n_current = len(pi)                        # neurons in this phase
        n_inactive = (pi <= 0.5).sum()             # inactive among current
        n_active   = n_current - n_inactive        # active  among current
        # sparsity / active-fraction wrt the ORIGINAL dense denominator
        snap[f"{lname}_sparsity"]    = float(n_inactive / dense_size)
        snap[f"{lname}_frac_active"] = float(n_active   / dense_size)
        # per-neuron stats (unchanged, independent of dense_size)
        snap[f"{lname}_mean_pi"]     = float(pi.mean())
        snap[f"{lname}_std_pi"]      = float(pi.std())
    return snap


# ─────────────────────────────────────────────────────────────────────────────
# 5.  PHASE 1 — BAYESIAN LOTTERY TICKET  (structured node-level IMP)
# ─────────────────────────────────────────────────────────────────────────────

def _row_l1_norms(layer: BayesianLinear,
                  mask: dict, layer_name: str) -> torch.Tensor:
    """
    L1 norm of each neuron's incoming weight vector (excluding bias column).
    If a mask exists, zero-out masked weights before computing the norm.
    Returns tensor of shape (out_features,).
    """
    mu = layer.mu.data[:, :-1].clone()           # (out, in) — weights only
    mu_key = f"{layer_name}.mu"
    if mu_key in mask:
        mu = mu * mask[mu_key][:, :-1].float()   # apply weight mask
    return mu.abs().sum(dim=1)                   # (out,)


def _apply_node_mask(model: RegressionNAP, node_mask: dict):
    """
    Zero out all incoming weights and bias for dead neurons.
    node_mask[layer_name] is a BoolTensor of shape (out_features,):
        True  = neuron alive
        False = neuron dead (all its rows zeroed)
    """
    with torch.no_grad():
        for layer_name in ["layer1", "layer2"]:
            if layer_name not in node_mask:
                continue
            alive = node_mask[layer_name]          # (out,)
            layer = getattr(model, layer_name)
            layer.mu.data[~alive, :]  = 0.0
            layer.rho.data[~alive, :] = -6.0       # σ → tiny
            layer.logit_pi.data[~alive] = -10.0    # π̃ → 0


def _count_alive(node_mask: dict) -> dict:
    return {k: int(v.sum().item()) for k, v in node_mask.items()}


def run_imp_regression(model: RegressionNAP,
                       X_train: torch.Tensor, y_train: torch.Tensor,
                       X_test:  torch.Tensor, y_test:  torch.Tensor,
                       prior_pi1: float, prior_pi2: float,
                       imp_rounds:       int   = 5,
                       prune_frac:       float = 0.25,
                       epochs_per_round: int   = 2_000,
                       lr:               float = 5e-3,
                       tau:              float = 0.5,
                       n_mc_train:       int   = 1,
                       n_mc_eval:        int   = 20,
                       ) -> tuple[RegressionNAP, dict, dict]:
    """
    Phase 1 — Bayesian Lottery Ticket for regression (structured, node-level).

    Each round:
      1. Train full-batch for `epochs_per_round` epochs (hard gate, S=1).
      2. Rank neurons in each hidden layer by L1 norm of their μ row.
      3. Prune the bottom `prune_frac` fraction of neurons per layer.
      4. Rewind surviving weights to epoch-0 init.

    Node-level (structured) pruning is used rather than weight-level so that
    `count_surviving_nodes` is exact and the prior recalibration in Phase 2
    is based on actual topology, not a weight-level approximation.

    Returns
    -------
    model      : rewound model with node_mask applied
    node_mask  : {layer_name: BoolTensor(out,)}  True = alive
    imp_history: dict for plotting
    """
    print(f"\n{'='*60}")
    print(f"PHASE 1 — Bayesian Lottery Ticket (IMP, {imp_rounds} rounds)")
    print(f"{'='*60}")

    n_data    = X_train.size(0)
    model.snapshot_epoch0()

    # initialise node mask: all neurons alive
    node_mask = {
        "layer1": torch.ones(model.layer1.out_features, dtype=torch.bool, device=DEVICE),
        "layer2": torch.ones(model.layer2.out_features, dtype=torch.bool, device=DEVICE),
    }

    imp_history = defaultdict(list)

    for rnd in range(1, imp_rounds + 1):
        alive = _count_alive(node_mask)
        print(f"\n  IMP round {rnd}/{imp_rounds}  "
              f"| alive: L1={alive['layer1']}  L2={alive['layer2']}")

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        # ── train ─────────────────────────────────────────────────────────────
        for epoch in range(1, epochs_per_round + 1):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            loss = -model.elbo(X_train, y_train,
                               prior_pi1, prior_pi2, n_data,
                               kl_weight=1.0, tau=tau, n_mc=n_mc_train)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            optimizer.step()
            # re-zero dead neurons after each step
            _apply_node_mask(model, node_mask)

            if epoch % 500 == 0:
                rmse = model.rmse(X_test, y_test, n_samples=n_mc_eval)
                print(f"    epoch {epoch:5d} | loss {loss.item():12.1f} "
                      f"| RMSE_test {rmse:.4f}")

        # ── record end-of-round stats ─────────────────────────────────────────
        rmse_end = model.rmse(X_test, y_test, n_samples=n_mc_eval)
        imp_history["round"].append(rnd)
        imp_history["rmse"].append(rmse_end)
        imp_history["alive_l1"].append(alive["layer1"])
        imp_history["alive_l2"].append(alive["layer2"])

        # ── structured node pruning ───────────────────────────────────────────
        for layer_name in ["layer1", "layer2"]:
            layer      = getattr(model, layer_name)
            alive_mask = node_mask[layer_name]
            norms      = _row_l1_norms(layer, {}, layer_name)  # (out,)

            # only rank currently alive neurons
            alive_idx  = alive_mask.nonzero(as_tuple=True)[0]
            n_alive    = alive_idx.numel()
            n_prune    = max(1, int(n_alive * prune_frac))
            # smallest norms among alive neurons → prune
            alive_norms = norms[alive_idx]
            _, sorted_local = torch.sort(alive_norms)
            prune_local = sorted_local[:n_prune]          # local indices
            prune_global = alive_idx[prune_local]         # global indices
            node_mask[layer_name][prune_global] = False

        new_alive = _count_alive(node_mask)
        print(f"  → after pruning: L1={new_alive['layer1']}  "
              f"L2={new_alive['layer2']}  RMSE={rmse_end:.4f}")

        # ── rewind to epoch-0 ─────────────────────────────────────────────────
        model.rewind_to_epoch0()
        _apply_node_mask(model, node_mask)

    # final round: train once more on the winning ticket before returning
    final_alive = _count_alive(node_mask)
    print(f"\n[IMP done] winning ticket: "
          f"L1={final_alive['layer1']}  L2={final_alive['layer2']}")
    rmse_ticket = model.rmse(X_test, y_test, n_samples=n_mc_eval)
    print(f"[IMP done] ticket RMSE (before SS-IG): {rmse_ticket:.4f}")

    return model, node_mask, dict(imp_history)


# ─────────────────────────────────────────────────────────────────────────────
# 6.  TRAINING  (SS-IG full-batch loop)
# ─────────────────────────────────────────────────────────────────────────────

def train_full_batch(model: RegressionNAP,
                     X_train: torch.Tensor, y_train: torch.Tensor,
                     X_test:  torch.Tensor, y_test:  torch.Tensor,
                     prior_pi1: float, prior_pi2: float,
                     epochs: int  = 10_000,
                     lr: float    = 5e-3,
                     kl_anneal_ep: int = 0,
                     tau: float   = 0.5,
                     n_mc_train: int  = 1,    # S=1 during training (thesis)
                     n_mc_eval:  int  = 50,   # more samples at evaluation
                     record_last: int = 1000, # collect stats from last N epochs
                     record_every: int = 10,  # step within last window
                     print_every: int  = 500,
                     node_mask: dict | None = None,  # ticket mask (Phase 2 only)
                     dense_size: int = 20,           # original hidden width (for sparsity)
                     ) -> dict:
    """
    Full-batch training loop for Simulation Study II.

    Records RMSE and sparsity for every `record_every`-th epoch
    within the last `record_last` epochs  →  100 checkpoints.

    Returns history dict.
    """
    n_data    = X_train.size(0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history   = defaultdict(list)

    record_start = epochs - record_last + 1   # epoch from which to start recording

    print(f"\n{'='*60}")
    print(f"Training  epochs={epochs}  lr={lr}  full-batch  S={n_mc_train}")
    print(f"Recording last {record_last} epochs every {record_every} → "
          f"{record_last // record_every} checkpoints")
    print(f"Prior π₁={prior_pi1:.2e}  π₂={prior_pi2:.2e}")
    print(f"{'='*60}\n")

    for epoch in range(1, epochs + 1):
        model.train()

        kl_w = 1.0 if kl_anneal_ep == 0 else min(1.0, epoch / kl_anneal_ep)

        optimizer.zero_grad(set_to_none=True)
        loss = -model.elbo(X_train, y_train,
                           prior_pi1, prior_pi2, n_data,
                           kl_weight=kl_w, tau=tau, n_mc=n_mc_train)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        # enforce ticket topology after every step (Phase 2 only)
        if node_mask is not None:
            _apply_node_mask(model, node_mask)

        # ── recording window ─────────────────────────────────────────────────
        in_window   = epoch >= record_start
        record_tick = (epoch - record_start) % record_every == 0

        if in_window and record_tick:
            rmse_train = model.rmse(X_train, y_train, n_samples=n_mc_eval)
            rmse_test  = model.rmse(X_test,  y_test,  n_samples=n_mc_eval)
            snap       = snapshot_sparsity(model, dense_size)

            history["epoch"].append(epoch)
            history["rmse_train"].append(rmse_train)
            history["rmse_test"].append(rmse_test)
            history["kl_w"].append(kl_w)
            history["sigma_eps"].append(model.sigma_eps.item())
            for k, v in snap.items():
                history[k].append(v)

        # ── console output ───────────────────────────────────────────────────
        if epoch % print_every == 0 or epoch == epochs:
            rmse_t = model.rmse(X_test, y_test, n_samples=20)
            sp1    = model.layer1.effective_sparsity(dense_size)
            sp2    = model.layer2.effective_sparsity(dense_size)
            print(f"  epoch {epoch:6d} | loss {loss.item():12.1f} "
                  f"| RMSE_test {rmse_t:.4f} "
                  f"| σ_ε={model.sigma_eps.item():.4f} "
                  f"| sp₁={sp1:.2%}  sp₂={sp2:.2%}")

    return dict(history)


# ─────────────────────────────────────────────────────────────────────────────
# 6.  POISSON-BINOMIAL LAYER SIZE SELECTION
# ─────────────────────────────────────────────────────────────────────────────

def poisson_binomial_pmf(pi_vec: np.ndarray) -> np.ndarray:
    """Hong (2013) DP recursion.  O(n²)."""
    n   = len(pi_vec)
    pmf = np.zeros(n + 1, dtype=np.float64)
    pmf[0] = 1.0
    for j, p in enumerate(pi_vec):
        for k in range(j + 1, 0, -1):
            pmf[k] = pmf[k] * (1 - p) + pmf[k - 1] * p
        pmf[0] *= (1 - p)
    return pmf


def select_optimal_layer_size(model: RegressionNAP,
                               node_mask: dict) -> dict:
    """
    PB selection over ALIVE neurons only.
    Dead neurons (logit_pi forced to -10 by _apply_node_mask) are excluded
    so their artificially zeroed π̃ does not trivially fix K^op = n_alive.
    """
    print(f"\n{'='*60}")
    print("PHASE 3 — Poisson-Binomial optimal layer size (alive neurons only)")
    print(f"{'='*60}")
    results = {}
    for layer_name in ["layer1", "layer2"]:
        layer      = getattr(model, layer_name)
        alive_mask = node_mask[layer_name].cpu()              # (out,) bool
        pi_all     = layer.pi_tilde.detach().cpu().numpy()

        # restrict to alive neurons
        pi_alive   = pi_all[alive_mask.numpy()].astype(np.float64)
        pi_alive   = np.clip(pi_alive, 1e-8, 1 - 1e-8)
        n_alive    = int(alive_mask.sum().item())

        pmf   = poisson_binomial_pmf(pi_alive)
        k_opt = max(1, int(np.argmax(pmf)))
        results[layer_name] = k_opt

        print(f"  {layer_name}: K^op={k_opt}  (of {n_alive} alive)  "
              f"sparsity over alive={1 - k_opt/n_alive:.2%}  "
              f"P(K={k_opt})={pmf[k_opt]:.6f}  "
              f"mean π̃ alive={pi_alive.mean():.4f}")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# 7.  NEURAL ATOM SELECTION + SPARSE NETWORK CONSTRUCTION
# ─────────────────────────────────────────────────────────────────────────────

def select_atoms(model: RegressionNAP,
                 k_optimal: dict,
                 node_mask: dict) -> dict:
    """
    Rank alive neurons by π̃_j and keep the top K^op_l.
    Dead neurons are excluded from ranking — they were already removed by IMP.
    """
    print(f"\n{'='*60}")
    print("PHASE 4 — Neural atom selection (among alive neurons)")
    print(f"{'='*60}")
    selected = {}
    for layer_name in ["layer1", "layer2"]:
        layer      = getattr(model, layer_name)
        alive_mask = node_mask[layer_name].cpu()              # (out,) bool
        pi_all     = layer.pi_tilde.detach().cpu()

        # global indices of alive neurons
        alive_idx  = alive_mask.nonzero(as_tuple=True)[0]     # (n_alive,)
        pi_alive   = pi_all[alive_idx]                         # (n_alive,)
        k_opt      = k_optimal[layer_name]

        # rank alive neurons by π̃ descending, keep top k_opt
        ranked_local  = torch.argsort(pi_alive, descending=True)
        chosen_local  = ranked_local[:k_opt]
        chosen_global = alive_idx[chosen_local].sort().values  # back to global idx

        selected[layer_name] = chosen_global

        min_sel = pi_alive[chosen_local].min().item()
        max_rej = (f"{pi_alive[ranked_local[k_opt:]].max().item():.4f}"
                   if k_opt < len(pi_alive) else "n/a")
        print(f"  {layer_name}: selected {k_opt} of {len(alive_idx)} alive atoms "
              f"| min π̃ selected={min_sel:.4f} "
              f"| max π̃ rejected={max_rej}")
    return selected


def build_sparse_regression_network(selected: dict,
                                    p_in: int = 5,
                                    sigma0: float = 1.0) -> RegressionNAP:
    k1 = len(selected["layer1"])
    k2 = len(selected["layer2"])
    sparse = RegressionNAP(p_in=p_in, hidden=k1, sigma0=sigma0).to(DEVICE)
    sparse.layer1 = BayesianLinear(p_in, k1, sigma0).to(DEVICE)
    sparse.layer2 = BayesianLinear(k1,   k2, sigma0).to(DEVICE)
    sparse.out    = nn.Linear(k2, 1).to(DEVICE)
    sparse.snapshot_epoch0()
    return sparse


# ─────────────────────────────────────────────────────────────────────────────
# 8.  UNCERTAINTY QUANTIFICATION  (regression)
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def compute_regression_uncertainty(model: RegressionNAP,
                                   X: torch.Tensor,
                                   y: torch.Tensor,
                                   n_samples: int = 200
                                   ) -> dict:
    """
    Predictive uncertainty decomposition for regression.

    Total predictive variance = Aleatoric + Epistemic
      Aleatoric : E_θ[σ_ε²]  — irreducible noise
      Epistemic : Var_θ[ŷ]   — uncertainty from weight posterior
    """
    model.eval()
    preds = model.predict_mc(X, n_samples=n_samples)  # (S, n, 1)
    preds = preds.squeeze(2)                           # (S, n)

    mean_pred  = preds.mean(0)                         # (n,)
    epistemic  = preds.var(0)                          # (n,)  Var_θ[ŷ]
    aleatoric  = (model.sigma_eps ** 2).expand(X.size(0))  # scalar → (n,)
    total_var  = epistemic + aleatoric                 # (n,)

    residuals  = (mean_pred - y.squeeze(1)).cpu().numpy()
    rmse       = float(np.sqrt((residuals**2).mean()))

    # 95% predictive interval coverage
    sigma_pred = total_var.sqrt()
    lower      = mean_pred - 1.96 * sigma_pred
    upper      = mean_pred + 1.96 * sigma_pred
    y_flat     = y.squeeze(1)
    coverage   = ((y_flat >= lower) & (y_flat <= upper)).float().mean().item()

    return dict(
        mean_pred  = mean_pred.cpu().numpy(),
        epistemic  = epistemic.cpu().numpy(),
        aleatoric  = aleatoric.cpu().numpy(),
        total_var  = total_var.cpu().numpy(),
        residuals  = residuals,
        rmse       = rmse,
        coverage95 = coverage,
    )


# ─────────────────────────────────────────────────────────────────────────────
# 9.  SUMMARY STATISTICS  (last-1000-epoch window)
# ─────────────────────────────────────────────────────────────────────────────

def summarise(history: dict, label: str = ""):
    """
    Report mean ± sd of RMSE and median sparsity over the
    recorded checkpoints (100 points from last 1000 epochs).
    """
    rmse  = np.array(history["rmse_test"])
    sp1   = np.array(history["layer1_sparsity"])
    sp2   = np.array(history["layer2_sparsity"])

    print(f"\n{'='*60}")
    print(f"SUMMARY  {label}")
    print(f"{'='*60}")
    print(f"  RMSE test  : {rmse.mean():.4f} ± {rmse.std():.4f}")
    print(f"  RMSE train : {np.array(history['rmse_train']).mean():.4f} "
          f"± {np.array(history['rmse_train']).std():.4f}")
    print(f"  Sparsity L1 (median) : {np.median(sp1):.4f}")
    print(f"  Sparsity L2 (median) : {np.median(sp2):.4f}")
    print(f"  Sparsity tuple       : ({np.median(sp1):.4f}, {np.median(sp2):.4f})")
    print(f"  σ_ε (final)          : {history['sigma_eps'][-1]:.4f}")
    return {
        "rmse_mean"  : rmse.mean(),
        "rmse_std"   : rmse.std(),
        "sp1_median" : np.median(sp1),
        "sp2_median" : np.median(sp2),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 10. PLOTTING
# ─────────────────────────────────────────────────────────────────────────────

def plot_results(imp_history:  dict,
                 ssig_history: dict,
                 ft_history:   dict,
                 uq:           dict,
                 X_test:       torch.Tensor,
                 y_test:       torch.Tensor,
                 k_optimal:    dict,
                 original_hidden: int,
                 save_dir:     str = ".") -> str:
    """
    Single 5-panel figure (GridSpec 3x2, Panel E spans full bottom row).
    A: RMSE vs Epochs (both models). B: Node sparsity vs Epochs (both models).
    C: Node sparsity vs IMP round.   D: Uncertainty decomposition.
    E: Regression reliability diagram.
    All sparsity relative to the SAME original dense architecture.
    """
    os.makedirs(save_dir, exist_ok=True)
    from scipy import stats as _stats

    fig = plt.figure(figsize=(14, 18), facecolor="white")
    gs  = GridSpec(3, 2, figure=fig,
                   hspace=0.42, wspace=0.32,
                   top=0.93, bottom=0.05, left=0.07, right=0.97)
    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[1, 0])
    ax_d = fig.add_subplot(gs[1, 1])
    ax_e = fig.add_subplot(gs[2, :])

    fig.suptitle(
        "NAP -- Simulation Study II  (Nonlinear Regression)\n"
        "y = 7x2/(1+x1^2) + sin(x3*x4) + 2x5 + e   |   FNN 5->20->20->1",
        fontsize=13, fontweight="bold"
    )

    def _style(ax, xlabel="", ylabel="", title=""):
        ax.set_facecolor(_C["bg"])
        ax.grid(True, color=_C["grid"], linewidth=0.7, linestyle="--", zorder=0)
        ax.spines[["top", "right"]].set_visible(False)
        if xlabel: ax.set_xlabel(xlabel, fontsize=9)
        if ylabel: ax.set_ylabel(ylabel, fontsize=9)
        if title:  ax.set_title(title, fontsize=10, fontweight="bold", pad=6)

    def _shade(ax, x, mu, std, color, alpha=0.15):
        ax.fill_between(np.asarray(x),
                        np.asarray(mu) - np.asarray(std),
                        np.asarray(mu) + np.asarray(std),
                        alpha=alpha, color=color, linewidth=0)

    C_S = _C["ssig"]; C_F = _C["ft"]; C_L1 = "#4C72B0"; C_L2 = "#E07B39"; C_R = _C["rmse"]

    tl1 = int(imp_history["alive_l1"][-1])
    tl2 = int(imp_history["alive_l2"][-1])
    k1  = k_optimal["layer1"]
    k2  = k_optimal["layer2"]

    def _sp_ssig():
        f1 = np.array(ssig_history["layer1_frac_active"])
        f2 = np.array(ssig_history["layer2_frac_active"])
        return (1 - f1*tl1/original_hidden).clip(0,1), (1 - f2*tl2/original_hidden).clip(0,1)

    def _sp_ft():
        f1 = np.array(ft_history["layer1_frac_active"])
        f2 = np.array(ft_history["layer2_frac_active"])
        return (1 - f1*k1/original_hidden).clip(0,1), (1 - f2*k2/original_hidden).clip(0,1)

    def _sp_imp():
        return ([1.0 - a/original_hidden for a in imp_history["alive_l1"]],
                [1.0 - a/original_hidden for a in imp_history["alive_l2"]])

    # ── Panel A: RMSE vs Epochs ──────────────────────────────────────────────
    _style(ax_a, "Epoch", "RMSE", "A -- RMSE vs Epochs")
    ep_s = np.array(ssig_history["epoch"])
    ax_a.plot(ep_s, ssig_history["rmse_test"],  color=C_S, lw=1.8, label="BLT+SS-IG test",  zorder=3)
    ax_a.plot(ep_s, ssig_history["rmse_train"], color=C_S, lw=1.1, ls="--", alpha=0.55, label="BLT+SS-IG train")
    if ft_history:
        ep_f = np.array(ft_history["epoch"])
        ax_a.plot(ep_f, ft_history["rmse_test"],  color=C_F, lw=1.8, label="(NAP) PB+FT test",  zorder=3)
        ax_a.plot(ep_f, ft_history["rmse_train"], color=C_F, lw=1.1, ls="--", alpha=0.55, label="(NAP) PB+FT train")
    ax_a.legend(fontsize=7, ncol=2)

    # ── Panel B: Node sparsity vs Epochs ─────────────────────────────────────
    _style(ax_b, "Epoch", "Node sparsity (wrt dense arch.)", "B -- Node Sparsity vs Epochs")
    ss1, ss2 = _sp_ssig()
    sd1 = np.array(ssig_history["layer1_std_pi"]) * tl1 / original_hidden
    sd2 = np.array(ssig_history["layer2_std_pi"]) * tl2 / original_hidden
    ax_b.plot(ep_s, ss1, color=C_S, lw=1.8, label="BLT+SS-IG L1")
    _shade(ax_b, ep_s, ss1, sd1, C_S)
    ax_b.plot(ep_s, ss2, color=C_S, lw=1.8, ls="--", label="BLT+SS-IG L2")
    _shade(ax_b, ep_s, ss2, sd2, C_S)
    if ft_history:
        ep_f = np.array(ft_history["epoch"])
        sf1, sf2 = _sp_ft()
        sdf1 = np.array(ft_history["layer1_std_pi"]) * k1 / original_hidden
        sdf2 = np.array(ft_history["layer2_std_pi"]) * k2 / original_hidden
        ax_b.plot(ep_f, sf1, color=C_F, lw=1.8, label="(NAP) PB+FT L1")
        _shade(ax_b, ep_f, sf1, sdf1, C_F)
        ax_b.plot(ep_f, sf2, color=C_F, lw=1.8, ls="--", label="(NAP) PB+FT L2")
        _shade(ax_b, ep_f, sf2, sdf2, C_F)
    ax_b.set_ylim(-0.02, 1.02)
    ax_b.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: "{:.0%}".format(v)))
    ax_b.legend(fontsize=7, ncol=2)

    # ── Panel C: Node sparsity vs IMP round ──────────────────────────────────
    _style(ax_c, "IMP Round", "Node sparsity (wrt dense arch.)",
           "C -- Node Sparsity vs IMP Round  (Bayesian IMP)")
    rounds = np.array(imp_history["round"])
    isp1, isp2 = _sp_imp()
    isp1 = np.array(isp1); isp2 = np.array(isp2)
    ax_c.plot(rounds, isp1, color=C_L1, marker="o", markersize=6, lw=2,
              label="Layer 1  (dense=" + str(original_hidden) + ")")
    ax_c.plot(rounds, isp2, color=C_L2, marker="s", markersize=6, lw=2, ls="--",
              label="Layer 2  (dense=" + str(original_hidden) + ")")
    for r, a1, a2, v1, v2 in zip(rounds, imp_history["alive_l1"], imp_history["alive_l2"], isp1, isp2):
        ax_c.annotate(str(a1), (r, v1), textcoords="offset points", xytext=(0,  7),
                      ha="center", fontsize=7.5, color=C_L1)
        ax_c.annotate(str(a2), (r, v2), textcoords="offset points", xytext=(0, -13),
                      ha="center", fontsize=7.5, color=C_L2)
    ax_c.set_ylim(-0.02, 1.05)
    ax_c.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: "{:.0%}".format(v)))
    ax_c.set_xticks(rounds)
    ax_c2 = ax_c.twinx()
    ax_c2.plot(rounds, imp_history["rmse"], color=C_R, marker="^",
               markersize=5, lw=1.4, ls=":", label="Test RMSE", zorder=4)
    ax_c2.set_ylabel("Test RMSE", fontsize=8, color=C_R)
    ax_c2.tick_params(axis="y", colors=C_R, labelsize=7)
    ax_c2.spines[["top", "right"]].set_visible(False)
    h1, l1_ = ax_c.get_legend_handles_labels()
    h2, l2_ = ax_c2.get_legend_handles_labels()
    ax_c.legend(h1 + h2, l1_ + l2_, fontsize=7, loc="upper left")

    # ── Panel D: Predictive uncertainty decomposition ─────────────────────────
    _style(ax_d, "Aleatoric (sigma_eps^2)", "Epistemic (Var[y_hat])",
           "D -- Uncertainty Decomposition  (NAP PB+FT)")
    rng   = np.random.default_rng(42)
    n_pts = len(uq["epistemic"])
    idx   = rng.choice(n_pts, size=min(1200, n_pts), replace=False)
    sc = ax_d.scatter(uq["aleatoric"][idx], uq["epistemic"][idx],
                      c=uq["total_var"][idx], cmap="viridis",
                      alpha=0.5, s=12, linewidths=0, zorder=3)
    cbar = fig.colorbar(sc, ax=ax_d, fraction=0.046, pad=0.02)
    cbar.set_label("Total predictive variance", fontsize=7.5)
    cbar.ax.tick_params(labelsize=7)
    info_d = ("Epistemic mean  = {:.4f}\n"
              "Aleatoric mean  = {:.4f}\n"
              "95% PI coverage = {:.2%}\n"
              "RMSE            = {:.4f}").format(
        uq["epistemic"].mean(), uq["aleatoric"].mean(),
        uq["coverage95"], uq["rmse"])
    ax_d.text(0.04, 0.97, info_d, transform=ax_d.transAxes, fontsize=7.5,
              verticalalignment="top",
              bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#CCC", alpha=0.9))

    # ── Panel E: Reliability diagram (full width) ─────────────────────────────
    _style(ax_e, "Nominal coverage (1-alpha)", "Empirical coverage",
           "E -- Regression Reliability Diagram  (NAP PB selection + Fine-tune)")
    nominal = np.linspace(0.01, 0.99, 99)
    emp_cov = []
    mp  = torch.from_numpy(uq["mean_pred"])
    tsd = torch.from_numpy(np.sqrt(uq["total_var"]))
    yfl = y_test.squeeze(1).cpu()
    for p in nominal:
        z   = _stats.norm.ppf(0.5 + p / 2)
        cov = ((yfl >= mp - z*tsd) & (yfl <= mp + z*tsd)).float().mean().item()
        emp_cov.append(cov)
    emp_cov = np.array(emp_cov)
    mace = float(np.abs(emp_cov - nominal).mean())
    ax_e.plot([0, 1], [0, 1], "k--", lw=1.3, label="Perfect calibration")
    ax_e.plot(nominal, emp_cov, color=C_F, lw=2.2, label="NAP sparse BNN")
    ax_e.fill_between(nominal, nominal, emp_cov, where=emp_cov >= nominal,
                      alpha=0.13, color=C_F,  label="Over-confident")
    ax_e.fill_between(nominal, nominal, emp_cov, where=emp_cov <  nominal,
                      alpha=0.13, color=C_R, label="Under-confident")
    info_e = "MACE = {:.4f}     95% PI coverage = {:.2%}".format(mace, uq["coverage95"])
    ax_e.text(0.02, 0.97, info_e, transform=ax_e.transAxes, fontsize=8.5,
              verticalalignment="top",
              bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#CCC", alpha=0.9))
    ax_e.set_xlim(0, 1); ax_e.set_ylim(0, 1)
    ax_e.set_aspect("equal")
    ax_e.legend(fontsize=8, loc="lower right")

    # ── save ─────────────────────────────────────────────────────────────────
    path = os.path.join(save_dir, "nap_simulation_II_results.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("[plot] saved -> " + path)
    return path


# ─────────────────────────────────────────────────────────────────────────────
# 12. FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_simulation_II(save_dir: str = "."):
    os.makedirs(save_dir, exist_ok=True)

    # ── hyperparameters ──────────────────────────────────────────────────────
    P_IN         = 5        # input dimension (x1…x5)
    HIDDEN       = 20       # neurons per hidden layer  (thesis: 20)
    SIGMA0       = 1.0
    EPOCHS       = 10_000   # epochs for SS-IG and fine-tune
    LR           = 5e-3     # thesis: 5 × 10⁻³
    TAU          = 0.5
    N_MC_TRAIN   = 1        # S=1 during training (thesis)
    N_MC_EVAL    = 50       # MC samples at evaluation
    RECORD_LAST  = 1000     # stats from last 1000 epochs …
    RECORD_EVERY = 10       # … every 10 → 100 checkpoints
    PRINT_EVERY  = 500
    L            = 2
    # IMP hyperparameters
    IMP_ROUNDS        = 5   # rounds of iterative magnitude pruning
    IMP_PRUNE_FRAC    = 0.25  # prune 20% of alive nodes per round
    IMP_EPOCHS_ROUND  = 1000 # training epochs per IMP round

    # ── data ─────────────────────────────────────────────────────────────────
    X_train, y_train, X_test, y_test = get_tensors()
    n_data = X_train.size(0)

    # ── dense-architecture prior (used for IMP phase) ─────────────────────────
    prior_pi1_dense = compute_prior_pi(P_IN,   HIDDEN, HIDDEN, n_data, L)
    prior_pi2_dense = compute_prior_pi(HIDDEN, HIDDEN, 1,      n_data, L)
    print(f"[prior dense] π₁={prior_pi1_dense:.2e}  π₂={prior_pi2_dense:.2e}")

    # ── build dense model ─────────────────────────────────────────────────────
    model = RegressionNAP(p_in=P_IN, hidden=HIDDEN, sigma0=SIGMA0).to(DEVICE)
    print(f"[model] {sum(p.numel() for p in model.parameters()):,} parameters  "
          f"| architecture: {P_IN}→{HIDDEN}→{HIDDEN}→1")

    # ─────────────────────────────────────────────────────────────────────────
    # PHASE 1 — Bayesian Lottery Ticket (IMP)
    # ─────────────────────────────────────────────────────────────────────────
    model, node_mask, imp_history = run_imp_regression(
        model, X_train, y_train, X_test, y_test,
        prior_pi1     = prior_pi1_dense,
        prior_pi2     = prior_pi2_dense,
        imp_rounds    = IMP_ROUNDS,
        prune_frac    = IMP_PRUNE_FRAC,
        epochs_per_round = IMP_EPOCHS_ROUND,
        lr            = LR,
        tau           = TAU,
        n_mc_train    = N_MC_TRAIN,
        n_mc_eval     = 20,
    )

    # surviving node counts and max L1 norms from the ticket
    # (used to recalibrate the prior for Phase 2)
    alive = {k: int(v.sum().item()) for k, v in node_mask.items()}
    b_norms = {}
    for lname in ["layer1", "layer2"]:
        mu = getattr(model, lname).mu.data[:, :-1]  # weights only
        alive_mask = node_mask[lname]
        mu_alive   = mu[alive_mask]
        b_norms[lname] = float(mu_alive.abs().sum(dim=1).max().item()) \
                         if mu_alive.numel() > 0 else 1.0

    print(f"\n[ticket] surviving nodes : L1={alive['layer1']}  L2={alive['layer2']}")
    print(f"[ticket] max L1 norms    : {b_norms}")

    # recalibrate prior to ticket dimensions
    prior_pi1_ticket = compute_prior_pi(
        P_IN, alive["layer1"], alive["layer2"], n_data, L,
        B_l=b_norms["layer1"])
    prior_pi2_ticket = compute_prior_pi(
        alive["layer1"], alive["layer2"], 1, n_data, L,
        B_l=b_norms["layer2"])
    print(f"[prior ticket] π₁={prior_pi1_ticket:.2e}  "
          f"π₂={prior_pi2_ticket:.2e}")

    # ─────────────────────────────────────────────────────────────────────────
    # PHASE 2 — SS-IG with soft Gumbel-Softmax on the winning ticket
    # ─────────────────────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("PHASE 2 — SS-IG soft gate on winning ticket (10 000 epochs)")
    print(f"{'='*60}")

    # rewind to epoch-0 before SS-IG
    model.rewind_to_epoch0()
    _apply_node_mask(model, node_mask)   # re-enforce ticket structure

    ssig_history = train_full_batch(
        model, X_train, y_train, X_test, y_test,
        prior_pi1    = prior_pi1_ticket,
        prior_pi2    = prior_pi2_ticket,
        epochs       = EPOCHS,
        lr           = LR,
        tau          = TAU,
        n_mc_train   = N_MC_TRAIN,
        n_mc_eval    = N_MC_EVAL,
        record_last  = RECORD_LAST,
        record_every = RECORD_EVERY,
        print_every  = PRINT_EVERY,
        node_mask    = node_mask,        # keep ticket mask active during SS-IG
        dense_size   = HIDDEN,           # sparsity always wrt original dense arch
    )

    ssig_summary = summarise(ssig_history,
                             "Phase 2 — SS-IG on ticket (last 1000 epochs)")

    # ─────────────────────────────────────────────────────────────────────────
    # PHASE 3 — Poisson-Binomial optimal layer size
    # ─────────────────────────────────────────────────────────────────────────
    k_optimal = select_optimal_layer_size(model, node_mask)

    # ─────────────────────────────────────────────────────────────────────────
    # PHASE 4 — Neural atom selection
    # ─────────────────────────────────────────────────────────────────────────
    selected  = select_atoms(model, k_optimal, node_mask)

    # ─────────────────────────────────────────────────────────────────────────
    # PHASE 5 — Fine-tune sparse BNN from epoch-0
    # ─────────────────────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("PHASE 5 — Fine-tune sparse BNN from epoch-0")
    print(f"{'='*60}")

    sparse_model = build_sparse_regression_network(
        selected, p_in=P_IN, sigma0=SIGMA0)

    k1 = k_optimal["layer1"]; k2 = k_optimal["layer2"]
    prior_pi1_ft = compute_prior_pi(P_IN, k1, k2, n_data, L)
    prior_pi2_ft = compute_prior_pi(k1,   k2, 1,  n_data, L)
    print(f"  Sparse-topology priors: π₁={prior_pi1_ft:.2e}  "
          f"π₂={prior_pi2_ft:.2e}")

    sparse_model.rewind_to_epoch0()

    ft_history = train_full_batch(
        sparse_model, X_train, y_train, X_test, y_test,
        prior_pi1    = prior_pi1_ft,
        prior_pi2    = prior_pi2_ft,
        epochs       = EPOCHS,
        lr           = LR,
        tau          = TAU,
        n_mc_train   = N_MC_TRAIN,
        n_mc_eval    = N_MC_EVAL,
        record_last  = RECORD_LAST,
        record_every = RECORD_EVERY,
        print_every  = PRINT_EVERY,
        dense_size   = HIDDEN,           # sparsity always wrt original dense arch
    )

    ft_summary = summarise(ft_history,
                           "Phase 5 — Fine-tune (last 1000 epochs)")

    # ─────────────────────────────────────────────────────────────────────────
    # Uncertainty quantification on final model
    # ─────────────────────────────────────────────────────────────────────────
    print("\n[UQ] computing predictive uncertainty on test set …")
    uq = compute_regression_uncertainty(
        sparse_model, X_test, y_test, n_samples=200)
    print(f"  RMSE              : {uq['rmse']:.4f}")
    print(f"  Epistemic mean    : {uq['epistemic'].mean():.4f}")
    print(f"  Aleatoric mean    : {uq['aleatoric'].mean():.4f}")
    print(f"  95% cov. coverage : {uq['coverage95']:.2%}")

    # ── Compression report ─────────────────────────────────────────────────
    k1t = alive["layer1"]; k2t = alive["layer2"]
    p_dense  = P_IN*HIDDEN + HIDDEN + HIDDEN**2 + HIDDEN + HIDDEN + 1
    p_ticket = P_IN*k1t   + k1t   + k1t*k2t   + k2t   + k2t   + 1
    p_sparse = P_IN*k1    + k1    + k1*k2     + k2    + k2    + 1
    print(f"\n[compression]")
    print(f"  Dense  : {P_IN}→{HIDDEN}→{HIDDEN}→1  "
          f"params={p_dense}")
    print(f"  Ticket : {P_IN}→{k1t}→{k2t}→1       "
          f"params={p_ticket}  compression={p_dense/max(p_ticket,1):.2f}×")
    print(f"  Sparse : {P_IN}→{k1}→{k2}→1         "
          f"params={p_sparse}  compression={p_dense/max(p_sparse,1):.2f}×")

    # ── Plots ─────────────────────────────────────────────────────────────
    plot_results(imp_history, ssig_history, ft_history,
                 uq, X_test, y_test, k_optimal,
                 original_hidden=HIDDEN, save_dir=save_dir)

    # ── Checkpoint ────────────────────────────────────────────────────────
    ckpt_path = os.path.join(save_dir, "nap_simulation_II.pt")
    torch.save({
        "state_dict"   : sparse_model.state_dict(),
        "k_optimal"    : k_optimal,
        "alive_ticket" : alive,
        "ssig_summary" : ssig_summary,
        "ft_summary"   : ft_summary,
        "uq"           : {k: v.tolist() if isinstance(v, np.ndarray) else v
                          for k, v in uq.items()},
    }, ckpt_path)
    print(f"[saved] checkpoint → {ckpt_path}")

    return sparse_model, ssig_summary, ft_summary, uq


# ─────────────────────────────────────────────────────────────────────────────
# 12. ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

SAVE_DIR = "./sim_II_results"

import sys as _sys
def _in_jupyter():
    return ("ipykernel" in _sys.modules or "IPython" in _sys.modules
            or any("jupyter" in a or "ipykernel" in a for a in _sys.argv))

if __name__ == "__main__" or _in_jupyter():
    print(f"\n{'='*60}")
    print(f"  NAP — Simulation Study II")
    print(f"  Device : {DEVICE}")
    print(f"{'='*60}\n")
    run_simulation_II(save_dir=SAVE_DIR)

[device] cuda

  NAP — Simulation Study II
  Device : cuda

[data] train=torch.Size([3000, 5])  test=torch.Size([1000, 5])
[prior dense] π₁=1.00e-06  π₂=1.00e-06
[model] 1,142 parameters  | architecture: 5→20→20→1

PHASE 1 — Bayesian Lottery Ticket (IMP, 5 rounds)

  IMP round 1/5  | alive: L1=20  L2=20
    epoch   500 | loss       6816.0 | RMSE_test 1.2045
    epoch  1000 | loss       6431.3 | RMSE_test 1.1640
  → after pruning: L1=15  L2=15  RMSE=1.1595

  IMP round 2/5  | alive: L1=15  L2=15
    epoch   500 | loss       6237.3 | RMSE_test 1.1589
    epoch  1000 | loss       6077.7 | RMSE_test 1.1425
  → after pruning: L1=12  L2=12  RMSE=1.1441

  IMP round 3/5  | alive: L1=12  L2=12
    epoch   500 | loss       8449.8 | RMSE_test 1.1652
    epoch  1000 | loss       5766.7 | RMSE_test 1.1816
  → after pruning: L1=9  L2=9  RMSE=1.1792

  IMP round 4/5  | alive: L1=9  L2=9
    epoch   500 | loss       5627.6 | RMSE_test 1.1678
    epoch  1000 | loss       5398.5 | RMSE_test 1.1612
  → 